## Calculating Energies

When we do long Caldeira Leggett simulations, it can become a lot of data to store if we want to keep track of the full wavefunction. To overcome this we can just keep track of specific parameters, such as the energy or location of the wavefunction.


In [ ]:
import numpy as np
from scipy.constants import Boltzmann  # type: ignore stubs
from slate_core import FundamentalBasis
from slate_core.metadata import (
    Domain,
    spaced_volume_metadata_from_stacked_delta_x,
)
from slate_core.plot import array_against_basis

from slate_quantum import operator, state
from slate_quantum.dynamics import (
    LangevinParameters,
    caldeira_leggett,
)
from slate_quantum.metadata import SpacedTimeMetadata

metadata = spaced_volume_metadata_from_stacked_delta_x((np.array([2 * np.pi]),), (21,))

potential_single = operator.build.cos_potential(metadata, 2)
potential = operator.build.repeat_potential(potential_single, (3,))
potential_metadata = potential.basis.inner.outer_recast.metadata()


mass = 1
hamiltonian = operator.build.kinetic_hamiltonian(potential, mass, hbar=1)
initial_state = state.build_coherent(potential_metadata, (0,), (0,), sigma_0=(0.5,))


parameters = LangevinParameters(mass=mass, lambda_=0, temperature=8 / Boltzmann, hbar=1)
times = FundamentalBasis(SpacedTimeMetadata(60, domain=Domain(delta=200 * np.pi)))


energies, norms = caldeira_leggett.solve_periodic_energies(
    initial_state,
    times,
    parameters,
    potential,
    method="Rouchon",
    target_delta=1e-1,
)

fig, ax, line = array_against_basis(norms[0, :])
ax.set_xlabel("Time / s")
ax.set_ylabel("Norm")
ax.set_title("Norm of the state")

fig, ax, line = array_against_basis(energies[0, :], measure="abs")
ax.set_xlabel("Time / s")
ax.set_ylabel("Energy")
ax.set_title("Energy of the state")